# Import Libraries

In [6]:
import pandas as pd 
import numpy as np
import os 
import requests
import json
from datetime import datetime
from zipfile import ZipFile

# Project Folders

In [38]:
from pathlib import Path

# Check current working directory if unsure
# print ("cwd:", Path.cwd())

# Set folder paths for project root, data, and outputs folders
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Ensure the folders exist: make the folders if they don't; skip if they do.

DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# Print and confirm 'absolute path'
print(DATA_DIR.resolve())

/Users/stephenhelvig/Documents/Python Projects/Citi Bike Strategy Dashboard/data


# Load Data

In [12]:
# establish where the files are and create a list
csv_files = sorted(DATA_DIR.glob("2022*.csv"))

# create a dictionary to tell pandas to force these columns to "string" dtypes when reading the .csv's
dtype_overrides = {
    "ride_id": "string",
    "start_station_id": "string",
    "end_station_id": "string",
}

# stack all monthly tables into one
df_trips = pd.concat(
    (pd.read_csv(f, dtype=dtype_overrides) for f in csv_files), 
    ignore_index=True
)

# Comments on above code
- glob finds all monthly 2022 CSVs
- generator reads one file at a time
- pd.concat(..., ignore_index=True) stacks into one dataframe and resets the index

In [17]:
# verify total number of files in the list and show the first 5
len(csv_files), csv_files[:5]

(36,
 [PosixPath('../data/202201-citibike-tripdata_1.csv'),
  PosixPath('../data/202201-citibike-tripdata_2.csv'),
  PosixPath('../data/202202-citibike-tripdata_1.csv'),
  PosixPath('../data/202202-citibike-tripdata_2.csv'),
  PosixPath('../data/202203-citibike-tripdata_1.csv')])

In [22]:
# number of rows and columns
df_trips.shape

(29838806, 13)

In [23]:
# verify dtypes for id columns
df_trips[["ride_id", "start_station_id", "end_station_id"]].dtypes

ride_id             string
start_station_id    string
end_station_id      string
dtype: object

In [21]:
df_trips.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,BFD29218AB271154,electric_bike,2022-01-21 13:13:43.392,2022-01-21 13:22:31.463,West End Ave & W 107 St,7650.05,Mt Morris Park W & W 120 St,7685.14,40.802117,-73.968181,40.804038,-73.945925,member
1,7C953F2FD7BE1302,classic_bike,2022-01-10 11:30:54.162,2022-01-10 11:41:43.422,4 Ave & 3 St,4028.04,Boerum Pl\t& Pacific St,4488.09,40.673746,-73.985649,40.688489,-73.991160,member
2,95893ABD40CED4B8,electric_bike,2022-01-26 10:52:43.096,2022-01-26 11:06:35.227,1 Ave & E 62 St,6753.08,5 Ave & E 29 St,6248.06,40.761227,-73.960940,40.745168,-73.986831,member
3,F853B50772137378,classic_bike,2022-01-03 08:35:48.247,2022-01-03 09:10:50.475,2 Ave & E 96 St,7338.02,5 Ave & E 29 St,6248.06,40.783964,-73.947167,40.745168,-73.986831,member
4,7590ADF834797B4B,classic_bike,2022-01-22 14:14:23.043,2022-01-22 14:34:57.474,6 Ave & W 34 St,6364.10,5 Ave & E 29 St,6248.06,40.749640,-73.988050,40.745168,-73.986831,member


# Wrangle Citibike date columns

In [24]:
# convert 'started_at' column into a datetime type (datetime64)
df_trips["started_at"] = pd.to_datetime(df_trips["started_at"], errors="coerce")

# create a 'date-only' column from the 'started_at' column
df_trips["date"] = df_trips["started_at"].dt.normalize() # normalize sets any time component to midnight without changing the date

# check the results
df_trips[["started_at", "date"]].head()

,started_at,date
0,2022-01-21 13:13:43.392,2022-01-21
1,2022-01-10 11:30:54.162,2022-01-10
2,2022-01-26 10:52:43.096,2022-01-26
3,2022-01-03 08:35:48.247,2022-01-03
4,2022-01-22 14:14:23.043,2022-01-22


In [25]:
# check parsing results
df_trips["started_at"].isna().mean()

np.float64(0.0)

In [26]:
# export to a new .csv
cbsd_main_2022_01_path = DATA_DIR / "cbsd_main_2022_01.csv"
df_trips.to_csv(cbsd_main_2022_01_path, index=False)
preview = pd.read_csv(cbsd_main_2022_01_path, nrows=5)
preview

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,date
0,BFD29218AB271154,electric_bike,2022-01-21 13:13:43.392,2022-01-21 13:22:31.463,West End Ave & W 107 St,7650.05,Mt Morris Park W & W 120 St,7685.14,40.802117,-73.968181,40.804038,-73.945925,member,2022-01-21
1,7C953F2FD7BE1302,classic_bike,2022-01-10 11:30:54.162,2022-01-10 11:41:43.422,4 Ave & 3 St,4028.04,Boerum Pl\t& Pacific St,4488.09,40.673746,-73.985649,40.688489,-73.991160,member,2022-01-10
2,95893ABD40CED4B8,electric_bike,2022-01-26 10:52:43.096,2022-01-26 11:06:35.227,1 Ave & E 62 St,6753.08,5 Ave & E 29 St,6248.06,40.761227,-73.960940,40.745168,-73.986831,member,2022-01-26
3,F853B50772137378,classic_bike,2022-01-03 08:35:48.247,2022-01-03 09:10:50.475,2 Ave & E 96 St,7338.02,5 Ave & E 29 St,6248.06,40.783964,-73.947167,40.745168,-73.986831,member,2022-01-03
4,7590ADF834797B4B,classic_bike,2022-01-22 14:14:23.043,2022-01-22 14:34:57.474,6 Ave & W 34 St,6364.10,5 Ave & E 29 St,6248.06,40.749640,-73.988050,40.745168,-73.986831,member,2022-01-22


# NOAA Weather Data for 2022

In [39]:
TOKEN = "PASTE_YOUR_TOKEN_HERE"

In [28]:
# API request from NOAA for daily average temperatue at LGA

# preparing the request
station_id = "GHCND:USW00014732"  # LaGuardia (LGA)

url = (
    "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
    "?datasetid=GHCND"
    "&datatypeid=TAVG"
    "&limit=1000"
    f"&stationid={station_id}"
    "&startdate=2022-01-01"
    "&enddate=2022-12-31"
)

# sending the request 
r = requests.get(url, headers={"token": TOKEN})
r.raise_for_status()
d = r.json() # parses the response ('r') into a Python dictionary

# filter the results to only entries where datatype is "TAVG"
avg = [item for item in d["results"] if item["datatype"] == "TAVG"]

# create a dataframe from the filtered list with 2 columns: date and tavg_c
df_wx = pd.DataFrame({
    "date": pd.to_datetime([item["date"] for item in avg], errors="coerce"),
    "tavg_c": [item["value"] / 10.0 for item in avg],  # tenths of °C -> °C
})

# Normalize to midnight so it matches df_trips["date"]
df_wx["date"] = df_wx["date"].dt.normalize()

df_wx.head(), df_wx.tail(), df_wx.shape

(        date  tavg_c
 0 2022-01-01    11.6
 1 2022-01-02    11.4
 2 2022-01-03     1.4
 3 2022-01-04    -2.7
 4 2022-01-05     3.2,
           date  tavg_c
 360 2022-12-27    -0.7
 361 2022-12-28     3.4
 362 2022-12-29     6.4
 363 2022-12-30     9.3
 364 2022-12-31     8.2,
 (365, 2))

In [31]:
# check parsing conversion results
df_wx["date"].isna().mean(), df_wx["date"].nunique()

(np.float64(0.0), 365)

In [32]:
# export weather data
cbsd_wx_2022_01_path = DATA_DIR / "cbsd_wx_2022_01.csv"
df_wx.to_csv(cbsd_wx_2022_01_path, index=False)
cbsd_wx_2022_01_path

PosixPath('../data/cbsd_wx_2022_01.csv')

# Merging "main" and "weather" Datasets

In [33]:
df_trips["date"].dtype, df_wx["date"].dtype

(dtype('<M8[us]'), dtype('<M8[us]'))

In [34]:
df_main_02 = df_trips.merge(df_wx, how="left", on="date", indicator=True)
df_main_02["_merge"].value_counts(dropna=False)

_merge
both          29838166
left_only          640
right_only           0
Name: count, dtype: int64

In [35]:
# see which dates are missing weather
missing_dates = (
    df_main_02.loc[df_main_02["_merge"] == "left_only", "date"]
    .value_counts()
    .head(20)
)
missing_dates

date
2021-12-31    583
2021-12-30      9
2021-12-29      4
2021-12-11      3
2021-11-13      3
2021-12-26      3
2021-12-12      2
2021-12-04      2
2021-11-17      2
2021-09-05      2
2021-09-13      1
2021-12-23      1
2021-11-16      1
2021-11-07      1
2021-07-22      1
2021-12-06      1
2021-11-09      1
2021-12-14      1
2021-12-28      1
2021-12-19      1
Name: count, dtype: int64

In [36]:
# filter to just 2022 dates
df_main_02_2022 = df_main_02[
    (df_main_02["date"] >= "2022-01-01") & (df_main_02["date"] <= "2022-12-31")
].copy()

df_main_02_2022["_merge"].value_counts(dropna=False)

_merge
both          29838166
left_only            0
right_only           0
Name: count, dtype: int64

In [37]:
# drop _merge and save .csv
df_main_02_2022 = df_main_02_2022.drop(columns=["_merge"])

cbsd_main_2022_02_path = DATA_DIR / "cbsd_main_2022_02.csv"
df_main_02_2022.to_csv(cbsd_main_2022_02_path, index=False)

cbsd_main_2022_02_path

PosixPath('../data/cbsd_main_2022_02.csv')